# ETF Price Simulation Demo

This notebook demonstrates the basic ETF price simulator using Geometric Brownian Motion (GBM).

## What is Geometric Brownian Motion?

Geometric Brownian Motion is a stochastic process used to model stock prices and other financial assets. It's described by the stochastic differential equation:

$$
dS = \mu S dt + \sigma S dW
$$

Where:
- $S(t)$ is the asset price at time $t$
- $\mu$ (mu) is the drift coefficient (expected return)
- $\sigma$ (sigma) is the volatility coefficient
- $dW$ is a Wiener process (Brownian motion)

For simulation, we discretize using the Euler-Maruyama method:

$$
S(t+dt) = S(t) \cdot \exp\left((\mu - 0.5\sigma^2)dt + \sigma\sqrt{dt} \cdot Z\right)
$$

Where $Z \sim N(0,1)$ is a standard normal random variable.

In [ ]:
# Add the backend directory to the path
import sys
from pathlib import Path

backend_path = Path().parent.parent / "backend"
sys.path.insert(0, str(backend_path))

import numpy as np
import matplotlib.pyplot as plt
from app.market.price_simulator import PriceSimulator, calculate_returns, calculate_volatility

## Basic Price Simulation

Let's create a simple price path with default parameters.

In [ ]:
# Create a simulator with default parameters
simulator = PriceSimulator(
    initial_price=100.0,
    drift=0.05,      # 5% annual return
    volatility=0.20, # 20% annual volatility
    time_step=0.01,  # ~3.65 days per step
    random_seed=42   # For reproducibility
)

# Generate a price path
num_steps = 252  # One trading year
price_path = simulator.generate_path(num_steps)

# Extract prices
prices = [tick.price for tick in price_path]
timestamps = [tick.timestamp for tick in price_path]

print(f"Generated {len(prices)} price points")
print(f"Initial price: ${prices[0]:.2f}")
print(f"Final price: ${prices[-1]:.2f}")
print(f"Total return: {(prices[-1]/prices[0] - 1)*100:.2f}%")

## Visualize the Price Path

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(timestamps, prices, linewidth=2, label='ETF Price')
plt.axhline(y=100.0, color='r', linestyle='--', alpha=0.5, label='Initial Price')
plt.xlabel('Time')
plt.ylabel('Price ($)')
plt.title('ETF Price Simulation using Geometric Brownian Motion')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Analyze Returns and Volatility

In [ ]:
# Calculate log returns
returns = calculate_returns(prices)

# Calculate realized volatility
realized_vol = calculate_volatility(returns, annualize=True)

print(f"Number of returns: {len(returns)}")
print(f"Mean daily return: {np.mean(returns)*100:.4f}%")
print(f"Std daily return: {np.std(returns)*100:.4f}%")
print(f"Realized annual volatility: {realized_vol*100:.2f}%")
print(f"Input volatility: {simulator.config.volatility*100:.2f}%")

## Distribution of Returns

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Time series of returns
ax1.plot(timestamps[1:], returns, linewidth=1)
ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax1.set_xlabel('Time')
ax1.set_ylabel('Log Return')
ax1.set_title('Returns Over Time')
ax1.grid(True, alpha=0.3)

# Histogram of returns
ax2.hist(returns, bins=30, edgecolor='black', alpha=0.7)
ax2.axvline(x=np.mean(returns), color='r', linestyle='--', label=f'Mean: {np.mean(returns)*100:.2f}%')
ax2.set_xlabel('Log Return')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of Returns')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Effect of Volatility

Let's compare different volatility regimes to see how they affect price behavior.

In [ ]:
volatilities = [0.10, 0.20, 0.40]  # 10%, 20%, 40% annual volatility
colors = ['blue', 'green', 'red']
labels = ['Low Vol (10%)', 'Medium Vol (20%)', 'High Vol (40%)']

plt.figure(figsize=(12, 6))

for vol, color, label in zip(volatilities, colors, labels):
    sim = PriceSimulator(
        initial_price=100.0,
        drift=0.05,
        volatility=vol,
        time_step=0.01,
        random_seed=42  # Same seed for fair comparison
    )
    path = sim.generate_path(252)
    prices = [tick.price for tick in path]
    timestamps = [tick.timestamp for tick in path]
    plt.plot(timestamps, prices, color=color, linewidth=2, label=label)

plt.xlabel('Time')
plt.ylabel('Price ($)')
plt.title('Price Paths with Different Volatilities')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Multiple Monte Carlo Paths

In practice, we run many simulations to understand the range of possible outcomes.

In [ ]:
num_simulations = 10
num_steps = 252

plt.figure(figsize=(12, 6))

for i in range(num_simulations):
    sim = PriceSimulator(
        initial_price=100.0,
        drift=0.05,
        volatility=0.20,
        time_step=0.01,
        random_seed=None  # Different seeds for each path
    )
    path = sim.generate_path(num_steps)
    prices = [tick.price for tick in path]
    timestamps = [tick.timestamp for tick in path]
    plt.plot(timestamps, prices, linewidth=1, alpha=0.6)

plt.xlabel('Time')
plt.ylabel('Price ($)')
plt.title(f'{num_simulations} Monte Carlo Price Paths')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Reproducibility with Random Seeds

A key feature for testing and research is reproducibility. Using the same seed should produce identical results.

In [ ]:
# First simulation
sim1 = PriceSimulator(
    initial_price=100.0,
    drift=0.05,
    volatility=0.20,
    time_step=0.01,
    random_seed=12345
)
path1 = sim1.generate_path(100)
prices1 = [tick.price for tick in path1]

# Second simulation with same seed
sim2 = PriceSimulator(
    initial_price=100.0,
    drift=0.05,
    volatility=0.20,
    time_step=0.01,
    random_seed=12345
)
path2 = sim2.generate_path(100)
prices2 = [tick.price for tick in path2]

# Check if they're identical
are_identical = np.allclose(prices1, prices2)
print(f"Paths are identical: {are_identical}")
print(f"First 10 prices from sim1: {[f'{p:.4f}' for p in prices1[:10]]}")
print(f"First 10 prices from sim2: {[f'{p:.4f}' for p in prices2[:10]]}")

## Step-by-Step Simulation

The simulator can also be used step-by-step, which is useful for:
- Interactive debugging
- Real-time simulation
- Integrating with other components

In [ ]:
sim = PriceSimulator(
    initial_price=100.0,
    drift=0.05,
    volatility=0.20,
    time_step=0.01,
    random_seed=42
)

print("Step-by-step simulation:")
print(f"Initial price: ${sim.get_current_price():.2f}")

for i in range(5):
    price = sim.step()
    print(f"Step {i+1}: ${price:.2f}")

print(f"\nCurrent step: {sim.get_current_step()}")
print(f"Current price: ${sim.get_current_price():.2f}")

# Reset and try again
sim.reset()
print(f"\nAfter reset:")
print(f"Current step: {sim.get_current_step()}")
print(f"Current price: ${sim.get_current_price():.2f}")

## Key Takeaways

1. **GBM Model**: The geometric Brownian motion model is a standard way to simulate asset prices, ensuring prices stay positive and capturing both trend (drift) and randomness (volatility).

2. **Volatility Matters**: Higher volatility leads to larger price swings, which is crucial for market makers to understand when setting spreads and managing risk.

3. **Reproducibility**: Using random seeds allows us to reproduce simulations exactly, which is essential for testing and comparing strategies.

4. **Monte Carlo**: Running multiple simulations helps us understand the range of possible outcomes and assess risk.

5. **Returns Analysis**: Log returns are preferred over simple returns because they're additive over time and follow a normal distribution under GBM.